# Companion Notebook 04: Positional Encodings Verification

This notebook verifies **Sinusoidal Positional Encodings** and **Rotary Positional Embeddings (RoPE)** calculations from scratch in PyTorch/NumPy.


## 1. 2D RoPE Rotation Hand-Calculation Verification
We run the 2D rotation of vector $\mathbf{x} = [1.0, 2.0]^T$ at position $m=1$ with $\theta = \pi/2$.


In [1]:
import numpy as np
import torch

# Define inputs
x = np.array([1.0, 2.0])
m = 1
theta = np.pi / 2  # 90 degrees

# Compute rotation matrix
cos_term = np.cos(m * theta)
sin_term = np.sin(m * theta)

R = np.array([[cos_term, -sin_term],
              [sin_term, cos_term]])

# Rotated vector
x_rot = R.dot(x)

print("Rotation Matrix R:")
print(R)
print("\nInput Vector x:", x)
print("Rotated Vector R * x:", x_rot)


Rotation Matrix R:
[[ 6.123234e-17 -1.000000e+00]
 [ 1.000000e+00  6.123234e-17]]

Input Vector x: [1. 2.]
Rotated Vector R * x: [-2.  1.]


## 2. Verification of Relative Distance Dot Product Preservation
We verify that for RoPE, the dot product of query $\mathbf{q}$ and key $\mathbf{k}$ depends only on their relative separation $n-m$.


In [2]:
# Helper function to rotate 2D vector by angle m * theta
def rotate_2d(vec, pos, theta_val):
    angle = pos * theta_val
    c, s = np.cos(angle), np.sin(angle)
    # Rotation matrix
    rot_mat = np.array([[c, -s],
                        [s, c]])
    return rot_mat.dot(vec)

# Let query and key be arbitrary 2D vectors
q_vec = np.array([1.5, -0.5])
k_vec = np.array([0.8, 1.2])
theta_base = 0.5

# Test case 1: m = 3, n = 5 (relative distance = 2)
q_3 = rotate_2d(q_vec, 3, theta_base)
k_5 = rotate_2d(k_vec, 5, theta_base)
dot_case1 = np.dot(q_3, k_5)

# Test case 2: m = 1, n = 3 (relative distance = 2)
q_1 = rotate_2d(q_vec, 1, theta_base)
k_3 = rotate_2d(k_vec, 3, theta_base)
dot_case2 = np.dot(q_1, k_3)

# Test case 3: Un-rotated dot product rotated by the difference (n - m = 2)
# R_(n-m) * k
k_diff = rotate_2d(k_vec, 2, theta_base)
dot_case3 = np.dot(q_vec, k_diff)

print("Dot product at m=3, n=5:", dot_case1)
print("Dot product at m=1, n=3:", dot_case2)
print("Dot product with relative rotation n-m=2:", dot_case3)
print("\nAre all three dot products identical (relative preservation)?")
print(np.allclose([dot_case1, dot_case2], dot_case3))


Dot product at m=3, n=5: -1.5270547830564887
Dot product at m=1, n=3: -1.5270547830564885
Dot product with relative rotation n-m=2: -1.5270547830564882

Are all three dot products identical (relative preservation)?
True


## 3. Sinusoidal Positional Encoding Implementation
We implement the static Sinusoidal Positional Encoding in PyTorch.


In [3]:
class SinusoidalPE(torch.nn.Module):
    def __init__(self, d_model, max_len=100):
        super(SinusoidalPE, self).__init__()
        # Create empty positional matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        # Division term matching formula
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-np.log(10000.0) / d_model))
        
        # Apply sin to even indices, cos to odd indices
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        
        self.register_buffer('pe', pe)
        
    def forward(self, seq_len):
        # Return slice of the positional matrix
        return self.pe[:seq_len, :]

# Instantiate PE layer
pe_layer = SinusoidalPE(d_model=8, max_len=10)
pe_output = pe_layer(seq_len=5)

print("Sinusoidal PE matrix shape:", pe_output.shape)
print("PE slice for first 2 tokens:\n", pe_output[:2, :].numpy())


Sinusoidal PE matrix shape: torch.Size([5, 8])
PE slice for first 2 tokens:
 [[0.0000000e+00 1.0000000e+00 0.0000000e+00 1.0000000e+00 0.0000000e+00
  1.0000000e+00 0.0000000e+00 1.0000000e+00]
 [8.4147096e-01 5.4030234e-01 9.9833414e-02 9.9500418e-01 9.9998331e-03
  9.9994999e-01 9.9999981e-04 9.9999952e-01]]
